In [ ]:
!pip install chromadb
!pip install -U -q "google-genai"

In [ ]:
from google import genai

# El cliente de Gemini para hacer los embedding
GEMINI_API_KEY = 
client = genai.Client(api_key=GEMINI_API_KEY)

## API de TMDB para conseguir información de películas y cast

In [ ]:
import json
import requests

# Para obtener información de una película
# usamos la API de TMDB
TMDB_API_KEY = 
TMDB_HEADERS = {
      "accept": "application/json",
      "Authorization": f"Bearer {TMDB_API_KEY}"
}

def get_movies_info(title):
  url = f"https://api.themoviedb.org/3/search/movie?query={title}&include_adult=false&language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_reviews(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/reviews?language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_cast(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?language=en-US"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_person_details(person_id):
  url = f"https://api.themoviedb.org/3/person/{person_id}"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)


## Añadir películas a la colección "movies" de ChromaDB

In [ ]:
import chromadb
chroma_client = chromadb.PersistentClient(path="chroma_db")

# Función para añadir películas encontradas a la colección
# la colección es una parte de la base de datos
# hay colecciones por categorías (películas, actores, reviews...)
def add_movies_to_collection(movies_info):
  movies_col = chroma_client.get_or_create_collection('movies')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  def get_cast_info(movie):
    cast_found = get_movie_cast(str(movie['id']))
    director = "";
    cast = "";

    for person in cast_found['cast']:
      if person['known_for_department'] == 'Directing':
        director += person['name'] + ", "
      else:
        cast += person['name'] + ", "
    return director, cast

  def get_metadata(movie, director, cast):
    return {
        "movie_title" : movie['title'],
        "director"    : director,
        "cast"        : cast,
        "popularity"  : movie['popularity'],
        "release_date": movie['release_date'],
        "vote_average": movie['vote_average'],
        "vote_count"  : movie['vote_count']
    }

  for movie in movies_info['results']:
    # Consultamos con nuestra colección
    result = movies_col.get(
      ids=[str(movie['id'])],
      include=[]
    )

    # Si no existe, la añade
    if not result['ids'] and movie['overview']:
      # Primero obtenemos al cast
      director, cast = get_cast_info(movie)

      content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                        contents=movie['overview']).embeddings[0]
      ids.append(str(movie['id']))
      embeddings.append(content_embeddings.values)
      metadatas.append(get_metadata(movie, director, cast))
      documents.append(movie['overview'])

  if len(ids) > 0:
    movies_col.add(
        ids = ids,
        embeddings = embeddings,
        metadatas = metadatas,
        documents = documents
    )

  print(f"Added {len(ids)} movies.")

# Ejemplo de cómo usarlo junto a la búsqueda en TMDB
movies = get_movies_info('mario the movie')
add_movies_to_collection(movies)

## Añadir reviews a la colección "reviews" de ChromaDB

In [ ]:
def add_reviews_to_collection(movie_id):
  reviews_col = chroma_client.get_or_create_collection('reviews')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  reviews = get_movie_reviews(movie_id)

  def get_metadata(review):
    if review["author_details"]["rating"]:
      return {
          "author"  : review["author"],
          "rating"  : review["author_details"]["rating"]
      }
    else:
      return {
          "author"  : review["author"],
      }

  for review in reviews['results']:

    # Consultamos con nuestra colección
    result = reviews_col.get(
      ids=[str(review['id'])],
      include=[]
    )
    # Si no existe, la añade
    if not result['ids'] and review['content']:
      ids.append(review['id'])
      metadatas.append(get_metadata(review))

      content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                        contents=review['content']).embeddings[0]
      embeddings.append(content_embeddings.values)
      documents.append(review['content'])

      if len(ids) > 0:
        reviews_col.add(
          ids = ids,
          embeddings = embeddings,
          metadatas = metadatas,
          documents = documents
        )

  print(f"Added {len(ids)} reviews.")

# Ejemplo con la película 99 de TMDB (Todo Sobre Mi Madre)
add_reviews_to_collection("99")


## Añadir actores a la colección "people" de ChromaDB

In [ ]:
def add_person_to_collection(person_id):
  people_col = chroma_client.get_or_create_collection('people')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  details = get_person_details(person_id)

  # Consultamos con nuestra colección
  result = people_col.get(
    ids=[str(details['id'])],
    include=[]
  )

  if not result['ids'] and details['biography']:
    ids.append(str(details['id']))

    gender = "Not set"
    if details['gender'] == 1:
      gender = "female"
    elif details['gender'] == 2:
      gender = "male"
    elif details['gender'] == 3:
      gender = "non binary"

    metadatas.append({'name': details['name'], 'department': details['known_for_department'], 'gender': gender})
    content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                      contents=details['biography']).embeddings[0]
    embeddings.append(content_embeddings.values)
    documents.append(details['biography'])

    if len(ids) > 0:
      people_col.add(
      ids = ids,
      embeddings = embeddings,
      metadatas = metadatas,
      documents = documents
    )

    print(f"Added {details['name']}")
  else:
    print("Person already exists in collection")

# Ejemplo con persona 31 (Tom Hanks)
add_person_to_collection("31")


## Ejemplo de consulta a la colección "movies" de ChromaDB

In [ ]:
# Ejemplo de hacerle una pregunta a la colección
query = "película sobre coches"

# Hay que hacer un embedding porque hemos no usamos el modelo
# nativo de chromadb, sino el de gemini al meterlos en la colección
query_embedding = client.models.embed_content(model="gemini-embedding-001",
                                              contents=query).embeddings[0]
movies_col = chroma_client.get_or_create_collection('movies')

res = movies_col.query(
    query_embeddings=[query_embedding.values],
    n_results=3
)

res['metadatas'][0]

<h2>Creacion de Agentes con Google Agent Development Kit</h2>


In [ ]:
# Instalar librerías necesarias
!pip install google-adk litellm -q

import os
import requests
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# API Key y configuración
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

print("Configuración ADK completada.")



In [ ]:
# Clase del Agente1: Investigador de cine
class AgenteInvestigadorCine:
    def __init__(self, tmdb_api_key, gemini_client, chroma_client):
        self.TMDB_HEADERS = {
            "accept": "application/json",
            "Authorization": f"Bearer {tmdb_api_key}"
        }
        self.client = gemini_client
        self.chroma_client = chroma_client

    # Buscar películas por título
    def buscar_peliculas_tmdb(self, titulo):
        url = f"https://api.themoviedb.org/3/search/movie?query={titulo}&include_adult=false&language=en-US&page=1"
        return requests.get(url, headers=self.TMDB_HEADERS).json()

    # Obtener director y cast
    def obtener_reparto_tmdb(self, movie_id):
        url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?language=en-US"
        data = requests.get(url, headers=self.TMDB_HEADERS).json()
        director = ", ".join([p['name'] for p in data.get('crew', []) if p.get('job') == "Director"])
        cast = ", ".join([p['name'] for p in data.get('cast', [])])
        return director, cast

    # Guardar películas en ChromaDB
    def guardar_peliculas_chroma(self, peliculas):
        movies_col = self.chroma_client.get_or_create_collection('movies')
        ids, embeddings, metadatas, documents = [], [], [], []

        for movie in peliculas.get('results', []):
            if not movie.get('overview'):
                continue

            # Evitar duplicados
            result = movies_col.get(ids=[str(movie['id'])], include=[])
            if result['ids']:
                continue

            director, cast = self.obtener_reparto_tmdb(movie['id'])
            emb = self.client.models.embed_content(
                model="gemini-embedding-001",
                contents=movie['overview']
            ).embeddings[0].values

            ids.append(str(movie['id']))
            embeddings.append(emb)
            metadatas.append({
                "movie_title": movie.get('title', 'Desconocido'),
                "director": director or 'Desconocido',
                "cast": cast or 'Desconocido',
                "popularity": movie.get('popularity', 0),
                "release_date": movie.get('release_date', 'Desconocida'),
                "vote_average": movie.get('vote_average', 0),
                "vote_count": movie.get('vote_count', 0)
            })
            documents.append(movie['overview'])

        if ids:
            movies_col.add(ids=ids, embeddings=embeddings, metadatas=metadatas, documents=documents)

        print(f"Agregado(s) {len(ids)} película(s) a ChromaDB")

    # Consultar películas en ChromaDB
    def consultar_peliculas_chroma(self, query, n_results=5):
        emb = self.client.models.embed_content(
            model="gemini-embedding-001",
            contents=query
        ).embeddings[0].values

        movies_col = self.chroma_client.get_or_create_collection('movies')
        res = movies_col.query(query_embeddings=[emb], n_results=n_results)

        flat_metadatas = [m for sublist in res.get('metadatas', []) for m in sublist]
        peliculas_info = []
        for m in flat_metadatas:
            peliculas_info.append({
                "titulo": m.get('movie_title', 'Sin título'),
                "director": m.get('director', 'Desconocido'),
                "cast": m.get('cast', 'Desconocido'),
                "fecha_estreno": m.get('release_date', 'Desconocida'),
                "popularidad": m.get('popularity', 0),
                "vote_average": m.get('vote_average', 0),
                "vote_count": m.get('vote_count', 0)
            })
        return peliculas_info


In [ ]:
# Crear instancia del agente
agente = AgenteInvestigadorCine(
    tmdb_api_key=TMDB_API_KEY,
    gemini_client=client,
    chroma_client=chroma_client
)

# Funciones que ADK puede llamar
def buscar_y_guardar_peliculas(titulo: str):
    pelis = agente.buscar_peliculas_tmdb(titulo)
    agente.guardar_peliculas_chroma(pelis)
    return f"Se agregaron {len(pelis.get('results', []))} películas de '{titulo}' a ChromaDB."

def obtener_peliculas_por_titulo(titulo: str):
    resultados = agente.consultar_peliculas_chroma(titulo, n_results=1)
    if resultados:
        return resultados[0]
    return f"No se encontró información sobre '{titulo}'."

def obtener_detalles_pelicula(titulo: str):
    resultados = agente.consultar_peliculas_chroma(titulo, n_results=1)
    if resultados:
        return {
            "titulo": resultados[0]["titulo"],
            "director": resultados[0]["director"],
            "cast": resultados[0]["cast"],
            "fecha_estreno": resultados[0]["fecha_estreno"]
        }
    return f"No se encontró información sobre '{titulo}'."


In [ ]:
# Crear agente con ADK
agente_cine = Agent(
    name="AgenteCine",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Agente experto en películas, consulta TMDB y ChromaDB",
    instruction=(
        "Eres un asistente experto en películas. "
        "Puedes buscar películas por título en TMDB y almacenarlas en ChromaDB, "
        "o consultar películas almacenadas según una descripción o título. "
        "Cuando el usuario pida buscar películas, usa 'buscar_y_guardar_peliculas'. "
        "Cuando el usuario pida consultar películas, usa 'obtener_peliculas_por_titulo' o 'obtener_detalles_pelicula'."
    ),
    tools=[buscar_y_guardar_peliculas, obtener_peliculas_por_titulo, obtener_detalles_pelicula]
)

print(f"Agente '{agente_cine.name}' creado con modelo '{MODEL_GEMINI_2_0_FLASH}'.")


In [ ]:
# Crear Runner y sesión
session_service = InMemorySessionService()
APP_NAME = "cine_app"
USER_ID = "Usuario"
SESSION_ID = "user"

import asyncio

# Crear sesión async
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session creada: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

runner = Runner(
    agent=agente_cine,
    app_name=APP_NAME,
    session_service=session_service
)
print(f"Runner creado para el agente '{runner.agent.name}'.")


In [ ]:
# Función para llamar al agente async
async def call_agent_async(query: str):
    print(f"\n>>> User Query: {query}")
    content = types.Content(role='user', parts=[types.Part(text=query)])
    final_response_text = "El agente no produjo respuesta final."

    async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text
            break

    print(f"<<< Agent Response: {final_response_text}")


In [20]:
#hablar con el agente
await call_agent_async("Busca la pelicula Oppenheimer")
await call_agent_async("Dime el reparto , director y fecha de estreno  de la pelicula Oppenheimer")



>>> User Query: Busca la pelicula Oppenheimer
Agregado(s) 15 película(s) a ChromaDB
<<< Agent Response: Ok. He añadido 20 películas de Oppenheimer a la base de datos.


>>> User Query: Dime el reparto , director y fecha de estreno  de la pelicula Oppenheimer
<<< Agent Response: El reparto de Oppenheimer es Cillian Murphy, Emily Blunt, Matt Damon, Robert Downey Jr., Florence Pugh, Josh Hartnett, Casey Affleck, Rami Malek, Kenneth Branagh, Benny Safdie, Jason Clarke, Dylan Arnold, Tom Conti, James D'Arcy, David Dastmalchian, Dane DeHaan, Alden Ehrenreich, Tony Goldwyn, Jefferson Hall, David Krumholtz, Matthew Modine, Scott Grimes, Kurt Koehler, John Gowans, Macon Blair, Harry Groener, Gregory Jbara, Ted King, Tim DeKay, Steven Houska, Petrie Willink, Matthias Schweighöfer, Alex Wolff, Josh Zuckerman, Rory Keane, Michael Angarano, Emma Dumont, Sadie Stratton, Britt Kyle, Guy Burnet, Tom Jenkins, Louise Lombard, Michael Andrew Baker, Jeff Hephner, Olli Haaskivi, David Rysdahl, Josh Peck, 